# Sleep Staging with HMMs: Supervised and Unsupervised Training (NBK 3)

This notebook trains two kinds of Gaussian hidden Markov model (HMM) on the features from NBK 1 and decodes held-out subjects with the Viterbi algorithm. The **supervised** HMM is fit by maximum likelihood from the expert labels. The **unsupervised** HMM is fit by Baum–Welch without any labels, and its states are matched to stages afterwards. Both are trained for K = 2, 3, 4, and 5 classes with EEG features, and for K = 2 and 5 with EEG and EOG features, using the same subject-level folds. We also train simpler models without transitions or with diagonal covariances, a learning curve, and a sensitivity check on the label-free window. Predictions and parameters are saved for evaluation.

**The model:** Each epoch $t$ has a hidden stage $z_t \in \{1, \dots, K\}$ and an observed feature vector $\mathbf{x}_t$ (5 EEG features, or 7 with EOG). For a sequence of $T$ epochs,

$$p(z_{1:T}, \mathbf{x}_{1:T}) = \pi_{z_1} \prod_{t=2}^{T} A_{z_{t-1} z_t} \prod_{t=1}^{T} \mathcal{N}(\mathbf{x}_t \mid \boldsymbol{\mu}_{z_t}, \boldsymbol{\Sigma}_{z_t}),$$

with start probabilities $\pi_k = P(z_1 = k)$, transition probabilities $A_{jk} = P(z_t = k \mid z_{t-1} = j)$, and a Gaussian with mean $\boldsymbol{\mu}_k$ and full covariance $\boldsymbol{\Sigma}_k$ for each stage. This encodes three assumptions:
1. The stage depends only on the previous stage (first-order Markov), so stage durations are geometric.
2. Given the stage, an epoch's features are independent of all other epochs and stages.
3. Within a stage, the features follow one multivariate Gaussian.

Each sequence is modeled separately, so no transition is counted across a gap or between recordings.

---

### Import statements

In [12]:
import os
import time
import json
import logging
import warnings
import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed
from scipy.optimize import linear_sum_assignment
from hmmlearn.hmm import GaussianHMM
from sklearn.metrics import cohen_kappa_score
from sklearn.mixture import GaussianMixture
import helpers

---

### Loading the processed data

We load the features from NBK 1 and split them into sequences, the unbroken runs of epochs that the HMM treats as separate chains. We check that the sequence lengths add up to the number of epochs and that no feature is missing.

In [13]:
features = helpers.load_features()
lengths = helpers.sequence_lengths(features)
assert lengths.sum() == len(features)
assert not features[helpers.FEATURE_SETS["EEG+EOG"]].isna().any().any()
print(f"{len(features):,} epochs in {len(lengths)} sequences, "
      f"from {features['recording'].nunique()} recordings")

170,252 epochs in 293 sequences, from 153 recordings


There are 170,252 epochs in 293 sequences from 153 recordings, the same as in NBK 1.

---

### Splitting by subject

We use 5-fold cross-validation over subjects, not epochs or recordings, so both nights of a subject are always in the same fold. The subjects are shuffled once with a fixed seed and cut into five folds. Each fold is held out once while the models are trained on the other four, so every subject is tested exactly once. The code checks that every subject appears in exactly one fold and that no subject is in both the training and test set.

In [14]:
folds = helpers.subject_folds(features["subject"])
assert sorted(sum(folds, [])) == sorted(features["subject"].unique())

rows = []
for k, test_subjects in enumerate(folds):
    test = features["subject"].isin(test_subjects)
    train_subjects = set(features.loc[~test, "subject"])
    assert not train_subjects & set(test_subjects), "a subject is in both train and test"
    rows.append({"fold": k, "train subjects": len(train_subjects), "test subjects": len(test_subjects),
                 "train epochs": (~test).sum(), "test epochs": test.sum()})
pd.DataFrame(rows).set_index("fold")

,train subjects,test subjects,train epochs,test epochs
fold,,,,
0,62,16,133444,36808
1,62,16,135660,34592
2,62,16,137414,32838
3,63,15,136201,34051
4,63,15,138289,31963


Each fold holds out 15 or 16 subjects (about 32,000 to 37,000 epochs) and trains on the other 62 or 63 (about 133,000 to 138,000 epochs). All models below use these same folds, so they are compared on the same held-out subjects.

---

### Defining the label sets for K = 2 to 5

For fewer than five classes, we merge stages in advance. K = 2 separates wake from sleep. K = 3 merges all non-REM stages. K = 4 merges the two light stages, N1 and N2, and keeps N3 as deep sleep. The supervised HMM has one state per class, and the unsupervised HMM has the same number of states, K.

In [15]:
rows = []
for K, classes in helpers.CLASSES.items():
    codes = helpers.collapse(features["stage"], K)
    for k, c in enumerate(classes):
        stages = [s for s in helpers.STAGES if helpers.MERGES[K][s] == c]
        rows.append({"K": K, "class": c, "stages": "+".join(stages),
                     "% of epochs": round(100 * (codes == k).mean(), 1)})
pd.DataFrame(rows).set_index(["K", "class"])

stages  % of epochs
K class                            
2 Wake               W         25.0
  Asleep  N1+N2+N3+REM         75.0
3 W                  W         25.0
  NREM        N1+N2+N3         59.8
  REM              REM         15.2
4 W                  W         25.0
  Light          N1+N2         52.3
  Deep              N3          7.6
  REM              REM         15.2
5 W                  W         25.0
  N1                N1         12.2
  N2                N2         40.0
  N3                N3          7.6
  REM              REM         15.2

Wake is 25.0% of epochs for every K. Merging makes some classes much larger: NREM is 59.8% at K = 3 and light sleep is 52.3% at K = 4, while N3 stays at 7.6%. Each label set has its own class balance, so results are best compared with Cohen's kappa rather than accuracy.

---

### Supervised HMM with maximum likelihood

When the stages are known, the log-likelihood splits into separate terms for the start probabilities, the transitions, and each stage's Gaussian. Each maximum likelihood estimate is then a count or an average over the training subjects:

$$\hat{\pi}_k = \frac{n^{\text{start}}_k + 1}{N_{\text{seq}} + K}, \qquad \hat{A}_{jk} = \frac{n_{jk} + 1}{\sum_{k'} n_{jk'} + K},$$

$$\hat{\boldsymbol{\mu}}_k = \frac{1}{n_k} \sum_{t:\, z_t = k} \mathbf{x}_t, \qquad \hat{\boldsymbol{\Sigma}}_k = \frac{1}{n_k} \sum_{t:\, z_t = k} (\mathbf{x}_t - \hat{\boldsymbol{\mu}}_k)(\mathbf{x}_t - \hat{\boldsymbol{\mu}}_k)^\top + 10^{-3} I.$$

Here $n^{\text{start}}_k$ counts sequences that start in stage $k$, $n_{jk}$ counts transitions from $j$ to $k$ within sequences, and $n_k$ counts epochs in stage $k$. We add 1 to every count (Laplace smoothing) so that no start or transition is impossible, which would otherwise make Viterbi rule out any path that uses it. The small $10^{-3} I$ keeps each covariance invertible. With these two additions, the estimates are smoothed maximum likelihood rather than exact.

In [19]:
def make_hmm(start, trans, means, covars):
    model = GaussianHMM(n_components=len(start), covariance_type="full", init_params="", params="")
    model.startprob_, model.transmat_ = start, trans
    model.n_features = means.shape[1]
    model.means_, model.covars_ = means, covars
    return model

def fit_supervised(X, y, lengths, K, diagonal=False):
    starts = y[np.append(0, np.cumsum(lengths)[:-1])]
    start = np.bincount(starts, minlength=K) + 1.0
    trans = helpers.transition_counts(y, lengths, K) + 1.0
    means = np.array([X[y == k].mean(axis=0) for k in range(K)])
    covars = np.array([np.cov(X[y == k].T, bias=True) for k in range(K)])
    if diagonal:
        covars = np.array([np.diag(np.diag(c)) for c in covars])
    covars += 1e-3 * np.eye(X.shape[1])
    return make_hmm(start / start.sum(), trans / trans.sum(axis=1, keepdims=True), means, covars)

X, y = features[helpers.EEG_FEATURES].to_numpy(), helpers.collapse(features["stage"], 5)
train = (~features["subject"].isin(folds[0])).to_numpy()
example = fit_supervised(X[train], y[train], helpers.sequence_lengths(features[train]), 5)
print("Transition matrix, fold 0 training subjects:")
pd.DataFrame(example.transmat_, index=helpers.STAGES, columns=helpers.STAGES).round(3)

Transition matrix, fold 0 training subjects:


,W,N1,N2,N3,REM
W,0.915,0.078,0.004,0.000,0.002
N1,0.091,0.710,0.164,0.001,0.034
N2,0.015,0.024,0.908,0.041,0.012
N3,0.010,0.005,0.174,0.810,0.001
REM,0.018,0.027,0.013,0.000,0.941


The transition matrix learned from fold 0's training subjects is close to the experts' transitions in NBK 2. Stages are persistent, N3 is left almost only for N2 (0.174, compared with 0.001 for REM), and wake leads mostly to N1. The supervised HMM therefore encodes the expected sleep cycle directly.

---

### Unsupervised HMM with Baum-Welch

Without labels, we maximize the likelihood of the features alone, $p(\mathbf{x}_{1:T}) = \sum_{z_{1:T}} p(z_{1:T}, \mathbf{x}_{1:T})$, with Baum–Welch, the expectation-maximization (EM) algorithm for HMMs. Each iteration has two steps:

- **E-step:** the forward-backward recursions compute how likely each state is at each epoch given the whole sequence,
$$\alpha_t(k) = \mathcal{N}(\mathbf{x}_t \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k) \sum_j \alpha_{t-1}(j) A_{jk}, \qquad \beta_t(j) = \sum_k A_{jk}\, \mathcal{N}(\mathbf{x}_{t+1} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)\, \beta_{t+1}(k),$$
$$\gamma_t(k) = P(z_t = k \mid \mathbf{x}_{1:T}) \propto \alpha_t(k)\, \beta_t(k), \qquad \xi_t(j, k) = P(z_{t-1} = j, z_t = k \mid \mathbf{x}_{1:T}).$$
- **M-step:** the same formulas as the supervised fit, with the counts replaced by expected counts, so that $\hat{A}_{jk} \propto \sum_t \xi_t(j, k)$ and $\hat{\boldsymbol{\mu}}_k = \sum_t \gamma_t(k)\, \mathbf{x}_t \,/\, \sum_t \gamma_t(k)$.

Each iteration can only increase the likelihood, but EM stops at a local maximum, so the starting point matters. We run **10 restarts** in each fold. One starts its means from k-means, and nine start from randomly chosen training epochs; the start and transition probabilities are drawn at random, and every covariance starts as the overall covariance of the features. A run stops when an iteration improves the total log-likelihood by less than 0.01, or after 500 iterations. We keep the restart with the highest final log-likelihood on the training subjects, which uses no labels. We use the `hmmlearn` library, which also adds a small constant (0.01) to each covariance in the M-step.

Before using real data, we check both fits on 4,000 epochs sampled from a known three-state HMM.

In [20]:
def one_restart(X, lengths, K, seed, kmeans):
    logging.getLogger("hmmlearn").setLevel(logging.CRITICAL)
    warnings.filterwarnings("ignore")
    rng = np.random.RandomState(seed)
    model = GaussianHMM(n_components=K, covariance_type="full", n_iter=500, tol=0.01,
                        random_state=rng, init_params="stmc" if kmeans else "stc")
    if not kmeans:
        model.means_ = X[rng.choice(len(X), K, replace=False)]   # random epochs as means
    began = time.time()
    model.fit(X, lengths)
    trace = np.array(model.monitor_.history) / len(X)            # log-likelihood per epoch
    hit_cap = len(trace) == 500 and (trace[-1] - trace[-2]) * len(X) > 0.01
    return {"model": model, "trace": trace.tolist(), "final": model.score(X, lengths) / len(X),
            "hit_cap": bool(hit_cap), "kmeans": kmeans, "seconds": time.time() - began}

def fit_unsupervised(X, lengths, K, seed, n_restarts=10):
    runs = Parallel(n_jobs=-1)(delayed(one_restart)(X, lengths, K, seed * 100 + r, r == 0)
                               for r in range(n_restarts))
    best = int(np.argmax([r["final"] for r in runs]))
    return runs[best]["model"], runs, best

true = make_hmm(np.array([0.6, 0.3, 0.1]),
                np.array([[0.90, 0.08, 0.02], [0.05, 0.90, 0.05], [0.02, 0.08, 0.90]]),
                np.array([[0.0, 0.0], [3.0, 0.0], [0.0, 3.0]]), np.tile(0.5 * np.eye(2), (3, 1, 1)))
X_syn, z_syn = true.sample(4000, random_state=0)
sup = fit_supervised(X_syn, z_syn, [4000], 3)
unsup, _, _ = fit_unsupervised(X_syn, [4000], 3, seed=0, n_restarts=3)
states = unsup.predict(X_syn)
overlap = pd.crosstab(states, z_syn).to_numpy()
print("Supervised: largest transition error", np.abs(sup.transmat_ - true.transmat_).max().round(3))
print("Unsupervised: states recovered", f"{overlap.max(axis=1).sum() / 4000:.1%}")

Supervised: largest transition error 0.014
Unsupervised: states recovered 99.5%


On the synthetic data, the supervised fit recovers every transition probability to within 0.014, and the unsupervised fit, matched by majority, recovers 99.5% of the true states. Both fits work when the data follow the model, which suggests that the fitting code is correct, so difficulties on real data are more likely to come from the data and the model's assumptions.

---

### Matching states to stages

The unsupervised model's states are numbered arbitrarily and have no stage names. After fitting, we decode the **training** subjects with Viterbi and count how often each state coincides with each expert class. We compare two ways of naming the states:
- **Majority:** each state takes the class it overlaps with most. Two states can share a class, and a class that is no state's majority is never predicted.
- **One-to-one:** the Hungarian algorithm finds the assignment that uses each class exactly once and agrees with the labels most often. Every class is predicted, even when a state mostly contains another class.

Majority is our main rule because it names each state after what it actually contains. The matching uses training subjects only and is fixed before any held-out subject is decoded.

In [21]:
def overlap_matrix(states, y, K):
    counts = np.zeros((K, K), dtype=int)
    np.add.at(counts, (states, y), 1)
    return counts

def match_majority(overlap):
    return overlap.argmax(axis=1)

def match_one_to_one(overlap):
    state, cls = linear_sum_assignment(-overlap)
    return cls[np.argsort(state)]

toy = np.array([[50, 1, 0], [40, 12, 1], [0, 3, 30]])
print("majority:  ", match_majority(toy))
print("one-to-one:", match_one_to_one(toy))

majority:   [0 0 2]
one-to-one: [0 1 2]


In the small example, two states overlap mostly with class 0. Majority matching names both of them class 0, so class 1 is never predicted. One-to-one matching gives the second state class 1, even though it contains more of class 0. The two rules can therefore give quite different results when states do not line up with classes.

---

### Viterbi decoding on held-out subjects

For each fold, we fit both models on the training subjects, freeze their parameters and the state matching, and decode every held-out sequence with the Viterbi algorithm. Viterbi finds the single most likely sequence of states, $\hat{z}_{1:T} = \arg\max_{z_{1:T}} p(z_{1:T} \mid \mathbf{x}_{1:T})$, by dynamic programming in log space:

$$\delta_t(k) = \log \mathcal{N}(\mathbf{x}_t \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k) + \max_j \left[\delta_{t-1}(j) + \log A_{jk}\right],$$

keeping a pointer to the best previous state and tracing back from the end. Because the transitions favor staying in the same stage, Viterbi can overrule a single ambiguous epoch when its neighbors clearly belong to one stage.

Fitting takes a few minutes per configuration, so fits are saved in `results/fits/` and reloaded if the features have the same number of epochs. That is the only check, so delete `results/fits/` after changing the features.

In [23]:
def run_config(data, feature_set, K, unsupervised=True, cache=None):
    if cache and os.path.exists(cache):
        saved = joblib.load(cache)
        if saved["n_epochs"] == len(data):
            print(f"{feature_set} K={K}: loaded saved fits")
            return saved["pred"], saved["info"]

    X = data[helpers.FEATURE_SETS[feature_set]].to_numpy()
    y = helpers.collapse(data["stage"], K)
    models = ["supervised"] + (["unsupervised", "unsupervised_1to1", "state"] if unsupervised else [])
    pred = {m: np.full(len(data), -1) for m in models}
    info = []

    for k, test_subjects in enumerate(folds):
        test = data["subject"].isin(test_subjects).to_numpy()
        train = ~test
        L_train, L_test = helpers.sequence_lengths(data[train]), helpers.sequence_lengths(data[test])

        sup = fit_supervised(X[train], y[train], L_train, K)
        pred["supervised"][test] = sup.predict(X[test], L_test)            # Viterbi
        fold = {"supervised": {"transmat": sup.transmat_, "means": sup.means_,
                               "test_ll": sup.score(X[test], L_test) / test.sum()}}
        if unsupervised:
            model, runs, best = fit_unsupervised(X[train], L_train, K, seed=helpers.SEED + k)
            overlap = overlap_matrix(model.predict(X[train], L_train), y[train], K)
            states = model.predict(X[test], L_test)
            pred["state"][test] = states
            pred["unsupervised"][test] = match_majority(overlap)[states]
            pred["unsupervised_1to1"][test] = match_one_to_one(overlap)[states]
            fold["unsupervised"] = {
                "transmat": model.transmat_, "means": model.means_, "overlap": overlap,
                "mapping": match_majority(overlap), "test_ll": model.score(X[test], L_test) / test.sum(),
                "best": best, "runs": [{q: r[q] for q in r if q != "model"} for r in runs]}
            iters = [len(r["trace"]) for r in runs]
            print(f"{feature_set} K={K} fold {k}: iterations {min(iters)} to {max(iters)}, "
                  f"{sum(r['hit_cap'] for r in runs)} hit the cap")
        info.append(fold)

    assert all((p >= 0).all() for p in pred.values())
    if cache:
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        joblib.dump({"n_epochs": len(data), "pred": pred, "info": info}, cache)
    return pred, info

predictions = features[["recording", "subject", "epoch", "stage"]].copy()
predictions["fold"] = predictions["subject"].map({s: k for k, fold in enumerate(folds) for s in fold})
params = {}

def run_and_store(feature_set, K):
    name = f"{feature_set}_K{K}"
    pred, params[name] = run_config(features, feature_set, K,
                                    cache=os.path.join(helpers.OUT, "fits", f"{name}.joblib"))
    for model, codes in pred.items():
        predictions[f"{name}_{model}"] = codes
    kappas = {m: cohen_kappa_score(helpers.collapse(features["stage"], K), c)
              for m, c in pred.items() if m != "state"}
    print(f"{name} kappa:", {m: round(v, 3) for m, v in kappas.items()})

for K in [2, 3, 4, 5]:
    run_and_store("EEG", K)

EEG K=2: loaded saved fits
EEG_K2 kappa: {'supervised': 0.757, 'unsupervised': 0.0, 'unsupervised_1to1': 0.454}
EEG K=3: loaded saved fits
EEG_K3 kappa: {'supervised': 0.718, 'unsupervised': 0.515, 'unsupervised_1to1': 0.584}
EEG K=4: loaded saved fits
EEG_K4 kappa: {'supervised': 0.671, 'unsupervised': 0.528, 'unsupervised_1to1': 0.523}
EEG K=5: loaded saved fits
EEG_K5 kappa: {'supervised': 0.642, 'unsupervised': 0.572, 'unsupervised_1to1': 0.489}


The supervised HMM scores a kappa of 0.757 at K = 2, falling to 0.718, 0.671, and 0.642 as classes are added. The unsupervised HMM is lower at every K: 0.515, 0.528, and 0.572 for K = 3 to 5, so it does best with five states.

At K = 2, the unsupervised kappa is exactly 0. Kappa is 0 when a model predicts one class for everything: both states overlap more with sleep, which is 75% of epochs, so majority matching names both "Asleep". One-to-one matching forces one state to be wake and reaches 0.454, but that is still well below the supervised model, so the two states the HMM finds do not line up with wake and sleep. One-to-one matching does better than majority at K = 3, about the same at K = 4, and worse at K = 5.

---

### EEG and EOG features

We repeat the K = 2 and K = 5 experiments with the two EOG features added, keeping everything else the same. Eye movements are part of how experts score REM and N1, so EOG may help separate them.

In [24]:
for K in [2, 5]:
    run_and_store("EEG+EOG", K)

EEG+EOG K=2: loaded saved fits
EEG+EOG_K2 kappa: {'supervised': 0.743, 'unsupervised': 0.242, 'unsupervised_1to1': 0.486}
EEG+EOG K=5: loaded saved fits
EEG+EOG_K5 kappa: {'supervised': 0.654, 'unsupervised': 0.575, 'unsupervised_1to1': 0.508}


With EOG, the supervised kappa rises from 0.642 to 0.654 at K = 5 but falls slightly at K = 2 (0.757 to 0.743). The unsupervised model barely changes at K = 5 (0.572 to 0.575). At K = 2 its kappa rises from 0 to 0.242, because in some folds one state is now matched to wake. EOG therefore helps a little with labels and makes little difference without them.

---

### Ablation: removing transitions and covariances

To see what each part of the model contributes, we train three simpler versions on the same folds, with EEG features:
- **No transitions, supervised:** the same Gaussians as the supervised HMM, but each epoch is classified on its own, using the class frequencies as the prior.
- **No transitions, unsupervised:** a Gaussian mixture model with K components and full covariances, fit by EM with 10 restarts and matched by majority.
- **Diagonal covariances:** the supervised HMM with every covariance between features set to 0.

In [ ]:
for K in [2, 3, 4, 5]:
    X, y = features[helpers.EEG_FEATURES].to_numpy(), helpers.collapse(features["stage"], K)
    pred = {m: np.full(len(features), -1) for m in ["classifier", "gmm", "diagonal"]}
    for k, test_subjects in enumerate(folds):
        test = features["subject"].isin(test_subjects).to_numpy()
        train = ~test
        L_train, L_test = helpers.sequence_lengths(features[train]), helpers.sequence_lengths(features[test])

        sup = fit_supervised(X[train], y[train], L_train, K)
        prior = np.bincount(y[train], minlength=K) / train.sum()
        pred["classifier"][test] = (sup._compute_log_likelihood(X[test]) + np.log(prior)).argmax(axis=1)

        gmm = GaussianMixture(K, n_init=10, init_params="random_from_data", max_iter=500,
                              random_state=helpers.SEED + k).fit(X[train])
        mapping = match_majority(overlap_matrix(gmm.predict(X[train]), y[train], K))
        pred["gmm"][test] = mapping[gmm.predict(X[test])]

        diag = fit_supervised(X[train], y[train], L_train, K, diagonal=True)
        pred["diagonal"][test] = diag.predict(X[test], L_test)
    for model, codes in pred.items():
        predictions[f"EEG_K{K}_{model}"] = codes
    print(f"K={K}", {m: round(cohen_kappa_score(y, c), 3) for m, c in pred.items()})

K=2 {'classifier': 0.677, 'gmm': 0.647, 'diagonal': 0.679}
K=3 {'classifier': 0.629, 'gmm': 0.484, 'diagonal': 0.612}
K=4 {'classifier': 0.609, 'gmm': 0.427, 'diagonal': 0.576}
K=5 {'classifier': 0.584, 'gmm': 0.456, 'diagonal': 0.551}


Transitions help the supervised model at every K, and the unsupervised model at K = 3 to 5. At K = 5, removing them lowers the supervised kappa from 0.642 to 0.584 and the unsupervised kappa from 0.572 to 0.456. Full covariances also matter: with diagonal covariances, the supervised kappa drops to 0.551 at K = 5 and is lower at every K, so the correlations between features carry information.

The exception is unsupervised K = 2, where the Gaussian mixture (0.647) does far better than the HMM (0). The mixture's two components line up much better with wake and sleep than the HMM's two states do.

---

### Learning curve

To see whether more data would help, we train both models at K = 5 on 5, 10, 20, and 40 training subjects and on all of them, testing each fold on the same held-out subjects at every size. The unsupervised model uses 5 restarts here to save time. We also record how long each fit takes.

In [26]:
curve_path = os.path.join(helpers.OUT, "learning_curve.csv")
X, y = features[helpers.EEG_FEATURES].to_numpy(), helpers.collapse(features["stage"], 5)
rows = []
for k, test_subjects in enumerate(folds):
    test = features["subject"].isin(test_subjects).to_numpy()
    L_test = helpers.sequence_lengths(features[test])
    train_subjects = np.random.RandomState(helpers.SEED + k).permutation(
        sorted(set(features["subject"]) - set(test_subjects)))
    for n in [5, 10, 20, 40, len(train_subjects)]:
        train = features["subject"].isin(train_subjects[:n]).to_numpy()
        L_train = helpers.sequence_lengths(features[train])
        began = time.time()
        sup = fit_supervised(X[train], y[train], L_train, 5)
        sup_seconds, began = time.time() - began, time.time()
        unsup, runs, _ = fit_unsupervised(X[train], L_train, 5, seed=helpers.SEED + k, n_restarts=5)
        unsup_seconds = time.time() - began
        mapping = match_majority(overlap_matrix(unsup.predict(X[train], L_train), y[train], 5))
        rows.append({"fold": k, "subjects": n if n <= 40 else "all", "epochs": train.sum(),
                     "supervised": cohen_kappa_score(y[test], sup.predict(X[test], L_test)),
                     "unsupervised": cohen_kappa_score(y[test], mapping[unsup.predict(X[test], L_test)]),
                     "supervised_seconds": sup_seconds, "unsupervised_seconds": unsup_seconds})
curve = pd.DataFrame(rows)
curve.to_csv(curve_path, index=False)
curve.groupby("subjects", sort=False).mean().drop(columns="fold").round(3)

,epochs,supervised,unsupervised,supervised_seconds,unsupervised_seconds
subjects,,,,,
5,11095.2,0.618,0.524,0.004,5.663
10,21595.0,0.637,0.566,0.003,9.751
20,43044.4,0.639,0.573,0.009,27.526
40,87410.8,0.642,0.555,0.014,58.195
all,136201.6,0.643,0.572,0.021,108.312


More subjects do not seem to help much. The supervised kappa is 0.618 with 5 subjects and 0.637 with 10, then levels off at about 0.64. The unsupervised kappa is 0.524 with 5 subjects and between 0.555 and 0.573 from 10 subjects on, without a clear trend. This suggests that the remaining errors come mostly from the model and features rather than a lack of data.

The supervised fit takes about 0.02 seconds even with all subjects, because it only counts and averages. The unsupervised fit grows with the data, from about 5 seconds with 5 subjects to about 108 seconds with all of them, because each EM iteration passes over every epoch.

---

### Sensitivity check: label-free window

The main trimming rule uses the expert labels to find each sleep period. We repeat the K = 5 experiment with EEG features on the label-free window from NBK 1 and compare the two. The two windows contain different epochs, so we also compare them on the epochs they share.

In [29]:
lights_off = helpers.load_features(os.path.join(helpers.PROC, "features_lights_off.csv.gz"))
pred_lo, _ = run_config(lights_off, "EEG", 5, cache=os.path.join(helpers.OUT, "fits", "lights_off_EEG_K5.joblib"))
y_lo = helpers.collapse(lights_off["stage"], 5)

table = lights_off[["recording", "epoch", "stage"]].copy()
for model in ["supervised", "unsupervised"]:
    table[model] = pred_lo[model]
shared = table.merge(predictions[["recording", "epoch", "EEG_K5_supervised", "EEG_K5_unsupervised"]],
                     on=["recording", "epoch"], validate="1:1")
y_shared = helpers.collapse(shared["stage"], 5)

rows = []
for model in ["supervised", "unsupervised"]:
    rows.append({"model": model,
                 "main window": cohen_kappa_score(helpers.collapse(predictions["stage"], 5),
                                                  predictions[f"EEG_K5_{model}"]),
                 "label-free window": cohen_kappa_score(y_lo, pred_lo[model]),
                 "main, shared epochs": cohen_kappa_score(y_shared, shared[f"EEG_K5_{model}"]),
                 "label-free, shared epochs": cohen_kappa_score(y_shared, shared[model])})
sensitivity = pd.DataFrame(rows).set_index("model")
sensitivity.to_csv(os.path.join(helpers.OUT, "sensitivity_lights_off.csv"))
print(f"{len(lights_off):,} epochs in the label-free window, {len(shared):,} shared with the main window")
sensitivity.round(3)

EEG K=5: loaded saved fits
174,007 epochs in the label-free window, 157,079 shared with the main window


,main window,label-free window,"main, shared epochs","label-free, shared epochs"
model,,,,
supervised,0.642,0.668,0.639,0.634
unsupervised,0.572,0.604,0.556,0.559


Both kappas are higher on the label-free window as a whole (0.668 and 0.604), likely because it contains different epochs, including more wake. On the 157,079 epochs the two windows share, the results are almost the same: 0.639 vs 0.634 for the supervised model and 0.556 vs 0.559 for the unsupervised model. Using the labels to set the window therefore does not change our conclusions.

---

### Saving outputs

`predictions.csv.gz` has one row per epoch with its recording, subject, epoch number, stage, and fold, and one column per model, named `features_K_model` (for example `EEG_K5_supervised`). Class codes follow `helpers.CLASSES`, and the `state` columns hold the unsupervised model's raw state numbers. `params.json` holds each fold's learned transition matrices, means, state matching, log-likelihood traces, and held-out log-likelihood.

In [32]:
def to_lists(x):
    if isinstance(x, dict):
        return {k: to_lists(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [to_lists(v) for v in x]
    if isinstance(x, np.ndarray):
        return x.tolist()
    return x.item() if isinstance(x, np.generic) else x

predictions.to_csv(os.path.join(helpers.OUT, "predictions.csv.gz"), index=False)
with open(os.path.join(helpers.OUT, "params.json"), "w") as f:
    json.dump(to_lists(params), f)
print(f"Saved predictions for {len(predictions):,} epochs and {predictions.shape[1] - 5} columns")
print("Saved parameters for", list(params))

Saved predictions for 170,252 epochs and 36 columns
Saved parameters for ['EEG_K2', 'EEG_K3', 'EEG_K4', 'EEG_K5', 'EEG+EOG_K2', 'EEG+EOG_K5']


We save predictions for all 170,252 epochs in 36 model columns, and the parameters of the six main configurations.

---